> **Archived (no gain, see plan_t2.md P1).** `models/ranking/context.py` was removed from the package, so this notebook no longer runs; the saved outputs are the record.

# Task 2 · P1: test-context features (plan_t2.md)

**Idea:** each pair holds one real item, and decoys are popularity-matched random items, so the
candidates in a user's *other* pairs are a noisy sample of what they read. New columns:
`ctx_sum / ctx_max / ctx_mean` = binary-cosine similarity of each candidate to those other
candidates (`models/ranking/context.py`), as a pairwise difference.

**Density:** the test holds out ~1% of interactions and validation ~10%, so validation context is
thinned with `keep = context_keep(...)` (≈ 0.1) to match the test.

**Ship bar:** ≥ +0.003 on the held-out half, on all seeds. Shipped baseline: val 0.9383, public 0.9410.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

from assignment_1.modules.models import (
    ItemSimilarity, binary_cosine, context_keep, context_pair_features,
)
from assignment_1.modules.tasks.task2 import task2_pair_features
from assignment_1.modules.util import load_csv, make_pair_validation

In [2]:
DATA_DIR = "../data"  # notebook runs from notebooks/

train = load_csv("tasks_1_2_train.csv", DATA_DIR)
test = load_csv("task_2_test.csv", DATA_DIR)

## C1. Coverage on the test
Upper bound on what P1 can change: rows whose user has at least one other test row.

In [3]:
def other_rows(pairs: pd.DataFrame) -> pd.Series:
    """Number of other rows of the same user, per row."""
    return pairs.groupby("user")["user"].transform("size") - 1

o = other_rows(test)
print(f"test rows with ≥1 other row: {(o > 0).mean():.1%}")
o.clip(upper=10).value_counts().sort_index().rename("rows (10 = 10+)")

test rows with ≥1 other row: 68.6%


user
0     6279
1     3996
2     2595
3     1672
4     1225
5      924
6      749
7      464
8      423
9      310
10    1363
Name: rows (10 = 10+), dtype: int64

## C2. Validation pairs and matched density

In [4]:
tr, pairs = make_pair_validation(train, seed=0)
y = (pairs["which"] == pairs["item1"]).astype(int).to_numpy()
half = np.random.default_rng(0).random(len(pairs)) < 0.5

keep = context_keep(train, test, tr, pairs)
ov = other_rows(pairs)
print(f"{len(pairs):,} validation pairs | keep = {keep:.4f}")
print(f"mean other rows: test {o.mean():.2f} | val raw {ov.mean():.2f} | val thinned ≈ {ov.mean() * keep:.2f}")
print(f"rows with ≥1 kept other row ≈ {(1 - (1 - keep) ** ov).mean():.1%} (test {(o > 0).mean():.1%})")

186,164 validation pairs | keep = 0.0967
mean other rows: test 3.00 | val raw 28.49 | val thinned ≈ 2.76
rows with ≥1 kept other row ≈ 66.2% (test 68.6%)


## C3. Features
Base = shipped Task 2 features (slow: two BPR fits). Context columns are cheap, so several
variants are built against one similarity fit on `tr`.

In [5]:
X_base = task2_pair_features(tr, pairs)
sim = ItemSimilarity(binary_cosine).fit(tr)

ctx = {
    "matched": context_pair_features(sim, pairs, keep=keep),
    "unthinned": context_pair_features(sim, pairs, keep=1.0),
    "2x matched": context_pair_features(sim, pairs, keep=min(1.0, 2 * keep)),
}
X_base.shape, {k: v.shape for k, v in ctx.items()}

((186164, 11),
 {'matched': (186164, 3), 'unthinned': (186164, 3), '2x matched': (186164, 3)})

## C4. Held-out accuracy
`ctx` column order: sum, max, mean. The **matched** row is the one that predicts the test;
**unthinned** is optimistic (10× more context than the test has).

In [6]:
def evaluate(X: np.ndarray) -> dict:
    out = {}
    for name, m in [("lr", LogisticRegression(max_iter=1000)),
                    ("gb", HistGradientBoostingClassifier(random_state=0))]:
        m.fit(X[half], y[half])
        out[name] = round((m.predict(X[~half]) == y[~half]).mean(), 4)
    return out

rows = {"base (shipped)": evaluate(X_base)}
for name, C in ctx.items():
    rows[f"+ ctx {name}"] = evaluate(np.column_stack([X_base, C]))
m = ctx["matched"]
rows["+ ctx matched: sum only"] = evaluate(np.column_stack([X_base, m[:, [0]]]))
rows["+ ctx matched: max only"] = evaluate(np.column_stack([X_base, m[:, [1]]]))
rows["+ ctx matched: sum, max"] = evaluate(np.column_stack([X_base, m[:, :2]]))
pd.DataFrame(rows).T

,lr,gb
base (shipped),0.9383,0.9380
+ ctx matched,0.9384,0.9382
+ ctx unthinned,0.9385,0.9387
+ ctx 2x matched,0.9384,0.9383
+ ctx matched: sum only,0.9383,0.9380
+ ctx matched: max only,0.9383,0.9384
"+ ctx matched: sum, max",0.9384,0.9382


## C5. Where the gain comes from
Held-out accuracy split by whether the row has any kept context (matched density).

In [7]:
Xm = np.column_stack([X_base, ctx["matched"]])
has_ctx = np.abs(ctx["matched"]).sum(axis=1) > 0
preds = {}
for name, X in [("base", X_base), ("+ ctx", Xm)]:
    lr = LogisticRegression(max_iter=1000).fit(X[half], y[half])
    preds[name] = lr.predict(X[~half]) == y[~half]
h = has_ctx[~half]
pd.DataFrame({
    name: {"with context": p[h].mean(), "no context": p[~h].mean(), "all": p.mean()}
    for name, p in preds.items()
}).assign(share=[h.mean(), (~h).mean(), 1.0]).round(4)

,base,+ ctx,share
with context,0.9377,0.9378,0.6229
no context,0.9394,0.9394,0.3771
all,0.9383,0.9384,1.0000


## C6. LR coefficients (matched)

In [8]:
lr = LogisticRegression(max_iter=1000).fit(Xm[half], y[half])
print("ctx_sum / ctx_max / ctx_mean coefficients:", lr.coef_[0][-3:].round(3))

ctx_sum / ctx_max / ctx_mean coefficients: [-0.128  0.405  0.081]


## C7. Robustness: other validation seeds (slow: two BPR fits per seed)
Ship only if `+ ctx` beats base on every seed.

In [9]:
seed_rows = []
for seed in [1, 2, 3]:
    tr_s, pairs_s = make_pair_validation(train, seed=seed)
    y_s = (pairs_s["which"] == pairs_s["item1"]).astype(int).to_numpy()
    half_s = np.random.default_rng(seed).random(len(pairs_s)) < 0.5
    Xb = task2_pair_features(tr_s, pairs_s)
    C = context_pair_features(ItemSimilarity(binary_cosine).fit(tr_s), pairs_s,
                              keep=context_keep(train, test, tr_s, pairs_s), seed=seed)
    for name, X in [("base", Xb), ("+ ctx", np.column_stack([Xb, C]))]:
        lr = LogisticRegression(max_iter=1000).fit(X[half_s], y_s[half_s])
        seed_rows.append({"seed": seed, "features": name,
                          "acc": (lr.predict(X[~half_s]) == y_s[~half_s]).mean()})
r = pd.DataFrame(seed_rows).pivot(index="seed", columns="features", values="acc")
r.assign(gain=r["+ ctx"] - r["base"]).round(4)

features,+ ctx,base,gain
seed,,,
1,0.9378,0.9377,0.0001
2,0.9371,0.9371,-0.0001
3,0.9373,0.9374,-0.0001


## Decision
- If `+ ctx matched` clears +0.003 on every seed: set `TASK2_TEST_CONTEXT = True` in
  `tasks/task2.py` (drop columns if C4 shows a subset is as good), then `make check`,
  `make changes`, and upload `dist/task_2_predictions.csv`.
- Record the result (val, public) in the Task 2 docstring and in plan_t2.md.